In [1]:
%cd /home/parthgandhi/Projects/SwingBot/swingbot/src

/home/parthgandhi/Projects/SwingBot/swingbot/src


In [2]:
from utils import setup_logger
from ingestion import fetch_nse_indices, fetch_nse_indices_data, fetch_nse_stocks_data
from ingestion.brokers.kite import KiteLogin, fetch_instruments
from ingestion.data_sources.nse import fetch_nse_industry_classification
from config.ingestion.brokers import KiteConfig

setup_logger(level=20)

# Testing NSE Indices Ingestion

In [3]:
nse_indices_df = fetch_nse_indices(download_flag=False)
nse_indices_df.head()

2026-03-14 16:57:15 | ERROR | ingestion.ingest | fetch_nse_indices | Failed for /home/parthgandhi/data/tmp/nse/indices/nse_index_nifty_reits_realty.csv. Error: found more fields than defined in 'Schema'

Consider setting 'truncate_ragged_lines=True'.
2026-03-14 16:57:15 | INFO | ingestion.ingest | fetch_nse_indices | Fetched Indices Data for 29/30


symbol,index_name,index_type
str,str,str
"""ABBOTINDIA""","""NIFTY HEALTHCARE""","""SECTOR"""
"""ALKEM""","""NIFTY HEALTHCARE""","""SECTOR"""
"""APOLLOHOSP""","""NIFTY HEALTHCARE""","""SECTOR"""
"""AUROPHARMA""","""NIFTY HEALTHCARE""","""SECTOR"""
"""BIOCON""","""NIFTY HEALTHCARE""","""SECTOR"""


# Testing KITE API Ingestion for Index

In [4]:
kite = KiteLogin(credentials_path=KiteConfig.CREDENTIALS_PATH)()
nse_kite_df = fetch_instruments(kite=kite, exchange="NSE").select(
    "instrument_token", "symbol", "name", "segment"
)

2026-03-14 16:57:19 | INFO | KiteLogin | _auto_login | Starting Kite Loign
2026-03-14 16:57:35 | INFO | KiteLogin | _generate_request_token | Request Token is Generated Successfully
2026-03-14 16:57:35 | INFO | KiteLogin | _generate_access_token | Access Token Generated is Successfully
2026-03-14 16:57:35 | INFO | KiteLogin | _auto_login | Kite Connection object created successfully
2026-03-14 16:57:36 | INFO | ingestion.brokers.kite.instruments | fetch_instruments | Successfully Fetched Instruments for NSE


In [6]:
import polars as pl

indices_lst = nse_indices_df.get_column("index_name").unique().to_list()

kite_indices = (
    nse_kite_df.filter(pl.col("name").is_in(indices_lst))
    # .get_column("name")
    # .to_list()
)

In [8]:
kite_indices.to_pandas()

,instrument_token,symbol,name,segment
0,256265,NIFTY 50,NIFTY 50,INDICES
1,260105,NIFTY BANK,NIFTY BANK,INDICES
2,260617,NIFTY 100,NIFTY 100,INDICES
3,259849,NIFTY IT,NIFTY IT,INDICES
4,261129,NIFTY REALTY,NIFTY REALTY,INDICES
5,261897,NIFTY FMCG,NIFTY FMCG,INDICES
6,262409,NIFTY PHARMA,NIFTY PHARMA,INDICES
7,262921,NIFTY PSU BANK,NIFTY PSU BANK,INDICES
8,263433,NIFTY AUTO,NIFTY AUTO,INDICES
9,263689,NIFTY METAL,NIFTY METAL,INDICES


In [ ]:
fetch_nse_indices_data(
    kite=kite,
    instruments_df=nse_kite_df,
    nse_indices_df=nse_indices_df,
    start_date="2025-01-01 00:00:00",
    end_date="2026-03-05 00:00:00",
    frequency="day",
)

# Testing KITE API Ingestion for Stocks

In [ ]:
fetch_nse_stocks_data(
    kite=kite,
    instruments_df=nse_kite_df.sample(200),
    start_date="2025-01-01 00:00:00",
    end_date="2026-03-05 00:00:00",
    frequency="day",
)

In [ ]:
fetch_nse_industry_classification(
    ins_df=nse_kite_df.sample(100), fetch_date="2026-03-05"
)